In [1]:
import numpy as np

np.random.seed(42)

# =========================================================
# WEEK 1 INPUTS
# =========================================================
week1_inputs = [
    np.array([0.211325, 0.788675]),
    np.array([0.723607, 0.276393]),
    np.array([0.166667, 0.500000, 0.833333]),
    np.array([0.125000, 0.375000, 0.625000, 0.875000]),
    np.array([0.875000, 0.625000, 0.375000, 0.125000]),
    np.array([0.150000, 0.350000, 0.550000, 0.750000, 0.950000]),
    np.array([0.900000, 0.100000, 0.700000, 0.300000, 0.500000, 0.800000]),
    np.array([0.111111, 0.222222, 0.333333, 0.444444, 0.555555, 0.666667, 0.777778, 0.888889]),
]

# =========================================================
# WEEK 2 INPUTS
# =========================================================
week2_inputs = [
    np.array([0.654299, 0.353479]),
    np.array([0.754299, 0.253479]),
    np.array([0.304299, 0.553479, 0.709691]),
    np.array([0.804299, 0.603479, 0.409691, 0.205016]),
    np.array([0.854299, 0.653479, 0.359691, 0.155016]),
    np.array([0.904299, 0.703479, 0.509691, 0.305016, 0.103275]),
    np.array([0.884299, 0.123479, 0.729691, 0.285016, 0.523275, 0.787492]),
    np.array([0.104299, 0.243479, 0.319691, 0.465016, 0.573275, 0.647492, 0.794889, 0.860861]),
]

# =========================================================
# WEEK 3 INPUTS
# =========================================================
week3_inputs = [
    np.array([0.582812, 0.421077]),
    np.array([0.712865, 0.284413]),
    np.array([0.118496, 0.481282, 0.876608]),
    np.array([1.000000, 0.683447, 0.334333, 0.000000]),
    np.array([0.882245, 0.615032, 0.380358, 0.114494]),
    np.array([0.000000, 0.226282, 0.564108, 0.905744, 1.000000]),
    np.array([0.905495, 0.091782, 0.689608, 0.305244, 0.491854, 0.804378]),
    np.array([0.113495, 0.214782, 0.338108, 0.437244, 0.549353, 0.673378, 0.771789, 0.898699]),
]

# =========================================================
# WEEK 1 OUTPUTS
# =========================================================
week1_outputs = np.array([
    1.1327056288856873e-125,
    0.5675786892822564,
    -0.032385408076210126,
    -17.20744048260943,
    60.223125,
    -1.3287857969718009,
    0.34356041660314957,
    9.0501517903694
])

# =========================================================
# WEEK 2 OUTPUTS
# =========================================================
week2_outputs = np.array([
    1.1867858443665185e-41,
    0.2715258567299176,
    -0.1198581070659559,
    -13.082213230390916,
    50.179390256321376,
    -1.5080002951000964,
    0.31639485635485903,
    9.0219949134074
])

# =========================================================
# WEEK 3 OUTPUTS
# =========================================================
week3_outputs = np.array([
    7.99676981308551e-19,
    0.5213723465552891,
    -0.04726977098841539,
    -25.67625344929702,
    64.78245026474816,
    -1.7372122723701597,
    0.3507828450585503,
    9.0587238074074
])

# =========================================================
# SIMPLE NEURAL NETWORK WITH SGD + BACKPROP
# One hidden layer, tanh activation, linear output
# =========================================================
class TinyNN:
    def __init__(self, input_dim, hidden_dim=6, lr=0.03, l2=1e-4):
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.lr = lr
        self.l2 = l2

        self.W1 = np.random.randn(input_dim, hidden_dim) * 0.2
        self.b1 = np.zeros((1, hidden_dim))
        self.W2 = np.random.randn(hidden_dim, 1) * 0.2
        self.b2 = np.zeros((1, 1))

    def forward(self, X):
        Z1 = X @ self.W1 + self.b1
        A1 = np.tanh(Z1)
        Z2 = A1 @ self.W2 + self.b2
        return Z1, A1, Z2

    def predict(self, X):
        _, _, yhat = self.forward(X)
        return yhat.ravel()

    def fit(self, X, y, epochs=4000):
        n = X.shape[0]
        y = y.reshape(-1, 1)

        for epoch in range(epochs):
            # SGD: shuffle each epoch
            idx = np.random.permutation(n)
            Xs = X[idx]
            ys = y[idx]

            for i in range(n):
                xi = Xs[i:i+1]
                yi = ys[i:i+1]

                # forward
                Z1, A1, yhat = self.forward(xi)

                # loss gradient (MSE)
                dY = 2.0 * (yhat - yi)

                # backprop
                dW2 = A1.T @ dY + self.l2 * self.W2
                db2 = dY

                dA1 = dY @ self.W2.T
                dZ1 = dA1 * (1 - np.tanh(Z1)**2)

                dW1 = xi.T @ dZ1 + self.l2 * self.W1
                db1 = dZ1

                # update
                self.W2 -= self.lr * dW2
                self.b2 -= self.lr * db2
                self.W1 -= self.lr * dW1
                self.b1 -= self.lr * db1

# =========================================================
# HELPER: standardize y for stable training
# =========================================================
def standardize_y(y):
    mu = np.mean(y)
    sigma = np.std(y)
    if sigma < 1e-8:
        sigma = 1.0
    y_std = (y - mu) / sigma
    return y_std, mu, sigma

# =========================================================
# HELPER: choose Week 4 candidate
# Strategy:
# - train tiny NN on Weeks 1-3
# - sample candidates near best observed point + some global points
# - pick highest predicted point
# - blend toward best observed point to reduce overfitting risk
# =========================================================
def propose_week4(X_hist, y_hist, hidden_dim=6, local_n=2500, global_n=800):
    dim = X_hist.shape[1]

    # best observed input so far
    best_idx = np.argmax(y_hist)
    x_best = X_hist[best_idx]

    # standardize outputs
    y_std, y_mu, y_sigma = standardize_y(y_hist)

    # fit NN
    model = TinyNN(input_dim=dim, hidden_dim=hidden_dim, lr=0.03, l2=1e-4)
    model.fit(X_hist, y_std, epochs=4000)

    # local candidates near best point
    local_noise = np.random.normal(loc=0.0, scale=0.08, size=(local_n, dim))
    local_candidates = np.clip(x_best + local_noise, 0, 1)

    # global candidates
    global_candidates = np.random.uniform(0, 1, size=(global_n, dim))

    # combine with historical points too
    candidates = np.vstack([local_candidates, global_candidates, X_hist])

    # score with NN
    pred_std = model.predict(candidates)
    pred = pred_std * y_sigma + y_mu

    # best predicted candidate
    x_nn = candidates[np.argmax(pred)]

    # blend toward best observed point to stay realistic
    x_week4 = 0.70 * x_nn + 0.30 * x_best
    x_week4 = np.clip(x_week4, 0, 1)

    return np.round(x_week4, 6), x_best, np.max(y_hist)

# =========================================================
# BUILD DATA PER FUNCTION
# =========================================================
all_week_inputs = [week1_inputs, week2_inputs, week3_inputs]
all_week_outputs = [week1_outputs, week2_outputs, week3_outputs]

week4_inputs = []

for fn_idx in range(8):
    X_hist = np.vstack([
        all_week_inputs[0][fn_idx],
        all_week_inputs[1][fn_idx],
        all_week_inputs[2][fn_idx]
    ])
    y_hist = np.array([
        all_week_outputs[0][fn_idx],
        all_week_outputs[1][fn_idx],
        all_week_outputs[2][fn_idx]
    ])

    x4, xbest, ybest = propose_week4(X_hist, y_hist)
    week4_inputs.append(x4)

# =========================================================
# DISPLAY RESULTS
# =========================================================
print("Suggested Week 4 inputs (arrays):")
for i, arr in enumerate(week4_inputs, start=1):
    print(f"Function {i}: {arr}")

print("\nPortal-ready format:")
for i, arr in enumerate(week4_inputs, start=1):
    formatted = "-".join(f"{x:.6f}" for x in arr)
    print(f"Function {i}: {formatted}")

Suggested Week 4 inputs (arrays):
Function 1: [0.183704 0.127166]
Function 2: [0.255353 0.749841]
Function 3: [0.094906 0.770362 0.859589]
Function 4: [0.820856 0.226975 0.784625 0.113609]
Function 5: [0.91786  0.273128 0.475575 0.052446]
Function 6: [0.052509 0.789636 0.354234 0.228337 0.889991]
Function 7: [0.833796 0.096195 0.232612 0.789158 0.185769 0.681185]
Function 8: [0.448564 0.335191 0.751694 0.277261 0.167614 0.717472 0.304117 0.767059]

Portal-ready format:
Function 1: 0.183704-0.127166
Function 2: 0.255353-0.749841
Function 3: 0.094906-0.770362-0.859589
Function 4: 0.820856-0.226975-0.784625-0.113609
Function 5: 0.917860-0.273128-0.475575-0.052446
Function 6: 0.052509-0.789636-0.354234-0.228337-0.889991
Function 7: 0.833796-0.096195-0.232612-0.789158-0.185769-0.681185
Function 8: 0.448564-0.335191-0.751694-0.277261-0.167614-0.717472-0.304117-0.767059
